<a href="https://colab.research.google.com/github/AKSHATKUAMRSINGH/AI_Career_Coach/blob/main/RAG_Chunking_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG LAB: The Science of Smart Document Processing
### Interactive 45-minute learning notebook

**Topic:** Text Chunking and Document Processing for Retrieval-Augmented Generation (RAG)  
**Format:** Self-paced, interactive notebook | **Duration:** 45 minutes  
**Audience:** Beginner Python learners

**What you will build:** A small document-retrieval prototype that demonstrates preprocessing, chunk size, overlap, metadata, and retrieval.

> This is a teaching simulator, not a production RAG system. It uses transparent TF-IDF lexical retrieval rather than paid APIs or an LLM. Production systems often use tokenizers, embeddings, vector databases, rerankers, and an LLM.

## Learning outcomes
By the end, you can:
1. Explain why long documents are split before retrieval.
2. Compare fixed-size chunking and overlapping chunks.
3. Explain chunk-size and overlap trade-offs.
4. Describe extraction, cleaning, chunking, metadata, and indexing.
5. Run a retrieval experiment and identify its limitations.
6. Propose a chunking strategy for a real document-Q&A application.

## 45-minute route

| Time | Section |
|---:|---|
| 0–4 min | The problem |
| 4–10 min | Chunking fundamentals |
| 10–19 min | Interactive chunking playground |
| 19–25 min | Document processing |
| 25–33 min | Retrieval simulator |
| 33–40 min | Hands-on challenge |
| 40–45 min | Knowledge check |

**How to use:** Run cells from top to bottom with Shift+Enter. Change sliders and text boxes to experiment. Google Colab is the easiest way to run the notebook.

## 1. The problem: why not give the whole document to an LLM? (4 min)

Imagine a student asks a university policy assistant: **“How many days do I have to submit an appeal after receiving a result?”**

The answer may be buried in a long handbook. Sending the entire document for every question can be inefficient, costly, and noisy. Models also have context-window limits.

**Retrieval-Augmented Generation (RAG)** finds relevant pieces of a knowledge source and supplies them to a language model as evidence.

1. Ingest documents.
2. Extract and clean text.
3. Split text into chunks.
4. Index chunks for search.
5. Retrieve passages relevant to a question.
6. Give the evidence to an LLM to generate an answer.

Chunking happens before indexing. Poor boundaries can separate a rule from its exception; overly large chunks may contain irrelevant material.

**Quick prediction:** If chunks are extremely small, what context might be lost? If they are extremely large, what noise might be retrieved?

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Install widgets with: %pip install ipywidgets")

In [ ]:
if WIDGETS_AVAILABLE:
    prediction = widgets.Textarea(
        placeholder="Very small chunks may..., while very large chunks may...",
        description="Prediction:", layout=widgets.Layout(width="100%", height="85px"))
    display(prediction)

**Think it through:** Very small chunks may omit surrounding context, definitions, or qualifications. Very large chunks may retrieve unrelated information and reduce precision. There is no universally perfect chunk size; it depends on the source documents, questions, retrieval method, and model context budget.

## 2. Chunking fundamentals (6 min)

A **chunk** is a segment of a document stored and retrieved as a unit.

Common strategies:
- **Fixed-size:** split after a chosen number of words or tokens.
- **Sentence-based:** group complete sentences.
- **Paragraph-based:** preserve paragraph boundaries.
- **Structure-aware:** split using headings, sections, tables, or layout.
- **Semantic:** split where the topic or meaning changes.

This lesson uses word-based splitting because it is easy to inspect. Real tokenizers count model-specific tokens, not words.

**Overlap** repeats part of one chunk at the beginning of the next. With chunk size 10 and overlap 2, chunks cover word positions 1–10, 9–18, 17–26. Overlap may preserve boundary context, but too much creates duplication and storage overhead.

`step size = chunk size − overlap`

Overlap must be smaller than chunk size.

In [ ]:
def split_into_chunks(text, chunk_size=40, overlap=8):
    """Educational word-based splitter; not a model tokenizer."""
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer.")
    if not isinstance(overlap, int) or overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        segment = words[start:start + chunk_size]
        if not segment:
            break
        chunks.append({
            "chunk_id": len(chunks),
            "start_word": start,
            "end_word": start + len(segment),
            "text": " ".join(segment)
        })
        if start + chunk_size >= len(words):
            break
    return chunks

demo_text = (
    "A university student may appeal an examination result within seven calendar days "
    "of the result publication date. The appeal must be submitted through the student "
    "portal with the course code, examination date, and a clear explanation. Late appeals "
    "are normally not accepted unless the student provides documented exceptional "
    "circumstances. The academic office acknowledges receipt within two working days. "
    "A review panel examines the submitted evidence and communicates its decision by email. "
    "An appeal does not guarantee a change in the original result."
)
for c in split_into_chunks(demo_text, 24, 5):
    print(f"Chunk {c['chunk_id']} | words {c['start_word']}–{c['end_word']-1}")
    print(c["text"], "\n")

### Inspect the boundaries

A sentence can be split across chunks. Overlap repeats words to help retain nearby context, but it does not guarantee complete sentence boundaries. Structure-aware splitting can be better for policies, manuals, research papers, and technical documentation.

## 3. Interactive Chunking Playground (9 min)

Change the source document, chunk size, and overlap. Observe the number of chunks, repeated word positions, boundaries, and text.

**Experiment:** First keep chunk size fixed and increase overlap. Then keep overlap fixed and increase chunk size.

In [ ]:
if WIDGETS_AVAILABLE:
    sample_document = widgets.Textarea(value=demo_text, description="Document:",
        layout=widgets.Layout(width="100%", height="140px"))
    size_slider = widgets.IntSlider(value=24, min=8, max=80, description="Chunk size", continuous_update=False)
    overlap_slider = widgets.IntSlider(value=5, min=0, max=20, description="Overlap", continuous_update=False)
    playground_out = widgets.Output()

    def update_playground(change=None):
        with playground_out:
            clear_output(wait=True)
            size = size_slider.value
            overlap = min(overlap_slider.value, size - 1)
            chunks = split_into_chunks(sample_document.value, size, overlap)
            total = len(sample_document.value.split())
            repeated = sum(len(c["text"].split()) for c in chunks) - total
            print(f"Source words: {total} | Chunks: {len(chunks)}")
            print(f"Approx. repeated word positions: {max(0, repeated)} | Step size: {size-overlap}")
            print("-" * 65)
            for c in chunks:
                print(f"CHUNK {c['chunk_id']} | positions {c['start_word']}–{c['end_word']-1}")
                print(c["text"], "\n")

    for control in (size_slider, overlap_slider, sample_document):
        control.observe(update_playground, names="value")
    display(sample_document, size_slider, overlap_slider, playground_out)
    update_playground()
else:
    print("Install ipywidgets to use the controls.")

### Mini investigation

Try: (1) size 12, overlap 0; (2) size 12, overlap 4; (3) size 40, overlap 4.

- Which creates the most chunks?
- Which repeats the most text?
- Which is most likely to keep a multi-sentence explanation together?

**Learning point:** smaller chunks can improve focus but lose context; larger chunks preserve context but may reduce precision. Overlap helps boundary continuity but increases duplication.

In [ ]:
if WIDGETS_AVAILABLE:
    experiment_log = widgets.Textarea(
        placeholder="Record what changed across the three settings...",
        description="Observations:", layout=widgets.Layout(width="100%", height="90px"))
    display(experiment_log)

## 4. Document processing before chunking (6 min)

Chunking is only one step. A practical pipeline:

1. **Extract** text from PDF, DOCX, HTML, or another source.
2. **Normalize** whitespace and extraction artefacts.
3. **Clean carefully**; remove repeated headers/footers only when they are truly noise.
4. **Preserve structure** such as headings, sections, tables, and page numbers.
5. **Chunk** into passages suitable for retrieval.
6. **Attach metadata** such as source, page, section, version, and permissions.
7. **Index** chunks for search.

Metadata supports citations, filtering, debugging, and access control.

In [ ]:
import re

def clean_extracted_text(raw_text):
    """Conservative normalization; preserve paragraph breaks and punctuation."""
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.rstrip() for line in text.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

raw_policy = (
    "STUDENT HANDBOOK — PAGE 4\n\n\n"
    "ASSESSMENT APPEALS\n\n"
    "A student may appeal a result within seven calendar days of publication.\n\n\n"
    "STUDENT HANDBOOK — PAGE 4"
)
cleaned_policy = clean_extracted_text(raw_policy)
print("BEFORE:", repr(raw_policy))
print("\nAFTER:\n", cleaned_policy)


In [ ]:
def make_chunk_records(text, source, section, page, chunk_size=35, overlap=7):
    return [
        {**chunk, "source": source, "section": section, "page": page}
        for chunk in split_into_chunks(text, chunk_size, overlap)
    ]

policy_records = make_chunk_records(
    cleaned_policy, "student_handbook.txt", "Assessment Appeals", 4,
    chunk_size=12, overlap=3)
for record in policy_records:
    print(record)

### Document-processing edge cases

- **PDF tables:** extraction may flatten columns; preserve row/column relationships.
- **Scanned PDFs:** OCR can introduce spelling and layout errors.
- **Headers/footers:** repeated page furniture can pollute search.
- **Code:** preserve indentation and code blocks.
- **Sensitive content:** enforce access permissions before indexing and retrieval.
- **Updates:** track document versions so stale chunks can be refreshed or removed.

The sample metadata is illustrative. In real ingestion, page and section values should come from the extractor and document structure, not be guessed.

## 5. Retrieval Simulator: which chunks answer the question? (8 min)

After chunking, a system finds passages relevant to a question. Production RAG systems often use dense embeddings and vector databases. This notebook uses **TF-IDF lexical retrieval** because it is lightweight and transparent.

TF-IDF weights informative terms. Cosine similarity ranks chunk vectors against a query vector.

**Limitation:** lexical retrieval may miss synonyms and paraphrases. “Appeal deadline” and “time allowed to challenge a grade” may mean similar things but share few exact terms. Embedding retrieval can help with semantic similarity, though it has its own failure modes.

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("Install scikit-learn: %pip install scikit-learn")

documents = [
    {"source":"student_handbook.txt","section":"Assessment Appeals","page":4,
     "text":"A student may appeal an examination result within seven calendar days of the result publication date. Submit the appeal through the student portal."},
    {"source":"student_handbook.txt","section":"Appeal Evidence","page":5,
     "text":"The appeal must include the course code, examination date, and a clear explanation. Documented exceptional circumstances may be considered for late appeals."},
    {"source":"student_handbook.txt","section":"Review Process","page":5,
     "text":"The academic office acknowledges receipt within two working days. A review panel examines evidence and communicates its decision by email."},
    {"source":"library_guide.txt","section":"Borrowing","page":2,
     "text":"Students may borrow up to five books for fourteen days. Renewals are available if no other student has reserved the item."},
    {"source":"it_support.txt","section":"Password Reset","page":1,
     "text":"To reset a university password, open the IT help portal and verify your identity using the registered email address."}
]

def retrieve_tfidf(query, records, top_k=3):
    if not SKLEARN_AVAILABLE:
        raise ImportError("Install scikit-learn to run retrieval.")
    texts = [r["text"] for r in records]
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(texts + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    ranked = sorted(enumerate(scores), key=lambda pair: pair[1], reverse=True)[:top_k]
    return [{**records[i], "score": float(score)} for i, score in ranked]

if SKLEARN_AVAILABLE:
    for result in retrieve_tfidf("How long after results are published can a student appeal?", documents):
        print(f"Score {result['score']:.3f} | {result['source']} | p.{result['page']} | {result['section']}")
        print(result["text"], "\n")

In [ ]:
if WIDGETS_AVAILABLE and SKLEARN_AVAILABLE:
    query_box = widgets.Text(value="What is the deadline to appeal an exam result?",
        description="Question:", layout=widgets.Layout(width="100%"))
    topk_slider = widgets.IntSlider(value=3, min=1, max=5, description="Top-k", continuous_update=False)
    retrieval_out = widgets.Output()

    def update_retrieval(change=None):
        with retrieval_out:
            clear_output(wait=True)
            results = retrieve_tfidf(query_box.value, documents, top_k=topk_slider.value)
            for rank, result in enumerate(results, start=1):
                print(f"#{rank} | similarity={result['score']:.3f} | {result['source']} | p.{result['page']} | {result['section']}")
                print(result["text"], "\n")
            if results and results[0]["score"] == 0:
                print("No lexical overlap found. Try rephrasing or consider semantic embeddings.")

    query_box.observe(update_retrieval, names="value")
    topk_slider.observe(update_retrieval, names="value")
    display(query_box, topk_slider, retrieval_out)
    update_retrieval()
else:
    print("Interactive retrieval requires ipywidgets and scikit-learn.")

### What should you inspect?

- **Relevance:** does the passage contain evidence that answers the question?
- **Recall:** did the retriever find the useful evidence?
- **Precision:** are the retrieved passages mostly useful or mostly noise?
- **Context completeness:** does the passage include qualifications or exceptions?
- **Provenance:** can the system point back to the original source and page?

Increasing `top_k` may improve recall but can add irrelevant context. Chunking and retrieval must be tuned together.

## 6. Hands-on challenge: tune a mini document-Q&A pipeline (7 min)

**Scenario:** A student asks: “Can I submit an appeal after seven days if I had an exceptional circumstance, and what must I include?”

Your task:
1. Select chunk size and overlap.
2. Retrieve the top passages.
3. Decide whether the evidence is sufficient.
4. Explain what metadata you would preserve.

There is no universally correct chunk size. Judge the retrieved evidence, not only the numeric similarity score.

In [ ]:
challenge_text = (
    "ASSESSMENT APPEALS. A student may appeal an examination result within seven calendar days "
    "of the result publication date. The appeal must be submitted through the student portal. "
    "The submission must include the course code, examination date, and a clear explanation. "
    "Late appeals are normally not accepted unless the student provides documented exceptional "
    "circumstances. The academic office acknowledges receipt within two working days. "
    "A review panel examines the submitted evidence and communicates its decision by email."
)
challenge_query = "Can a late appeal be accepted for exceptional circumstances and what details are required?"
for c in split_into_chunks(challenge_text, 28, 8):
    print(f"Chunk {c['chunk_id']}: {c['text']}\n")

In [ ]:
if WIDGETS_AVAILABLE:
    challenge_size = widgets.IntSlider(value=28, min=10, max=60, description="Chunk size", continuous_update=False)
    challenge_overlap = widgets.IntSlider(value=8, min=0, max=20, description="Overlap", continuous_update=False)
    challenge_output = widgets.Output()

    def run_challenge(change=None):
        with challenge_output:
            clear_output(wait=True)
            overlap = min(challenge_overlap.value, challenge_size.value - 1)
            chunks = split_into_chunks(challenge_text, challenge_size.value, overlap)
            print(f"Generated {len(chunks)} chunks.")
            if SKLEARN_AVAILABLE:
                records = [{"source":"student_handbook.txt","section":"Assessment Appeals",
                            "page":4, **chunk} for chunk in chunks]
                results = retrieve_tfidf(challenge_query, records, top_k=min(3, len(records)))
                print("\nTop evidence:")
                for result in results:
                    print(f"Score {result['score']:.3f} | chunk {result['chunk_id']} | p.{result['page']}")
                    print(result["text"], "\n")
            print("Check: do the passages include BOTH the exception and required fields?")

    challenge_size.observe(run_challenge, names="value")
    challenge_overlap.observe(run_challenge, names="value")
    display(challenge_size, challenge_overlap, challenge_output)
    run_challenge()

In [ ]:
if WIDGETS_AVAILABLE:
    design_response = widgets.Textarea(
        placeholder="I chose ... because ... . The evidence is/is not sufficient because ... . I would preserve ... metadata and test using ...",
        description="Your design:", layout=widgets.Layout(width="100%", height="120px"))
    display(design_response)

**Example of a strong design explanation:** “I would start with paragraph- or heading-aware chunks, then test a moderate token budget with a small overlap. The answer requires both the late-appeal exception and mandatory submission fields. I would verify that retrieved context contains both. I would preserve document name, page, section, version, and permission metadata. I would evaluate with representative student questions and measure whether supporting evidence appears in the top results.”

These are starting principles, not universal settings.

## 7. Knowledge check (5 min)

Answer each question, then click **Check my answers**. Aim for at least 4/5.

In [ ]:
quiz_questions = [
    {"q":"1. What is the main purpose of chunking in RAG?",
     "options":["Make documents look shorter","Split content into manageable units for indexing and retrieval","Train an LLM from scratch","Remove all metadata"],
     "answer":1,"why":"Chunks are the units indexed and retrieved as evidence."},
    {"q":"2. What does overlap do?",
     "options":["Repeats content between neighboring chunks","Removes duplicate documents","Converts text into embeddings","Guarantees perfect retrieval"],
     "answer":0,"why":"Overlap can preserve context across boundaries, but increases duplication."},
    {"q":"3. A likely risk of chunks that are too large is:",
     "options":["They always lose all context","They cannot contain multiple sentences","They may include irrelevant information and reduce precision","They eliminate retrieval"],
     "answer":2,"why":"Large chunks can bring unrelated content along with a relevant passage."},
    {"q":"4. Why attach source and page metadata?",
     "options":["For decoration","For provenance, citations, filtering, and debugging","To increase chunk word count","To replace evaluation"],
     "answer":1,"why":"Metadata traces evidence to its source and supports filtering."},
    {"q":"5. A limitation of TF-IDF in this notebook is:",
     "options":["It retrieves only images","It understands every synonym","It may miss semantically similar wording","It requires a paid LLM API"],
     "answer":2,"why":"TF-IDF relies on term overlap; embeddings can capture some paraphrases."}
]

if WIDGETS_AVAILABLE:
    quiz_controls = []
    for item in quiz_questions:
        display(widgets.HTML(value=f"<b>{item['q']}</b>"))
        control = widgets.RadioButtons(options=list(enumerate(item["options"])),
                                       layout=widgets.Layout(width="100%"))
        quiz_controls.append(control)
        display(control)

    quiz_button = widgets.Button(description="Check my answers", button_style="primary")
    quiz_output = widgets.Output()

    def grade_quiz(_):
        with quiz_output:
            clear_output()
            score = 0
            for i, item in enumerate(quiz_questions):
                selected = quiz_controls[i].value
                correct = selected == item["answer"]
                score += int(correct)
                print(("Correct" if correct else "Not quite — correct: " + item["options"][item["answer"]]))
                print("Why: " + item["why"] + "\n")
            print(f"FINAL SCORE: {score}/5")
            print("Great work!" if score >= 4 else "Review the lesson and try again.")

    quiz_button.on_click(grade_quiz)
    display(quiz_button, quiz_output)
else:
    print("Install ipywidgets to use the interactive quiz.")

## 8. Real-world applications and final takeaways

### Where this concept is used
- **University assistants:** handbooks, course guides, policies.
- **Enterprise search:** internal procedures and documentation.
- **Customer support:** troubleshooting manuals.
- **Legal/compliance research:** clauses with page provenance.
- **Research assistants:** methods, findings, and definitions across papers.

### Production architecture

```text
PDF / DOCX / HTML
       |
Text extraction + OCR (if needed)
       |
Cleaning + structure preservation
       |
Chunking + metadata
       |
Embedding model → vector database / search index
       |
User question → query processing → retrieve top-k chunks
       |
Optional reranking + context assembly
       |
LLM generates a grounded answer with citations
```

### Five takeaways
1. Chunking determines the units the retriever can find.
2. Chunk size trades focused retrieval against sufficient context.
3. Overlap can preserve boundary context but causes duplication.
4. Structure and metadata matter as much as chunk length.
5. Evaluate with real questions and inspect evidence; do not tune by intuition alone.

**Exit ticket:** How can a chunking decision change the quality of an answer generated by a RAG application? A complete answer mentions context, retrieval relevance, boundary effects, and a trade-off.

### Optional extensions
- Compare word-, sentence-, and paragraph-based splitting.
- Replace TF-IDF with sentence-transformer embeddings.
- Add reranking and measure whether evidence improves.
- Load a public-domain document and preserve page citations.

### Completion checklist
- [ ] I can explain why RAG uses chunks.
- [ ] I can explain chunk size and overlap.
- [ ] I experimented with at least three settings.
- [ ] I understand metadata and document structure.
- [ ] I ran the retrieval simulator.
- [ ] I completed the challenge and quiz.

**End of lesson — RAG LAB**